In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import output
    limpiar = output.clear
except ImportError:
    def limpiar():
        print("\n" * 3)

try:
    from IPython.display import display
except ImportError:
    display = print

In [ ]:
VICTORIA = 100000
ESCALA = 1000
K_SAT = 4
HEURISTICAS = ["h1", "h2", "h3", "h4", "h5"]

# h1 centro | h2 ataque 4+1 | h3 bloqueo 4+1 | h4 ataque 3+2 | h5 prevención 3+2
PESOS_CONFIG_1 = {"h1": 1, "h2": 4, "h3": 4, "h4": 2, "h5": 2}    # Balanceada
PESOS_CONFIG_2 = {"h1": 3, "h2": 5, "h3": 1, "h4": 4, "h5": 0.5}  # Agresiva

In [ ]:
COLORES = {"Rojo": 1, "Amarillo": 2}
SIMBOLOS = {0: "▪️", 1: "🔴", 2: "🟡"}
_IDX = {}


def indices_ventanas(n_filas, n_cols):
    clave = (n_filas, n_cols)
    if clave not in _IDX:
        ventanas = []
        for f in range(n_filas):
            for c in range(n_cols):
                for df, dc in ((0, 1), (1, 0), (1, 1), (-1, 1)):
                    if 0 <= f + 4 * df < n_filas and 0 <= c + 4 * dc < n_cols:
                        ventanas.append([(f + i * df) * n_cols + c + i * dc for i in range(5)])
        _IDX[clave] = np.array(ventanas)
    return _IDX[clave]


def saturar(n):
    return n / (n + K_SAT)


class Tablero:
    def __init__(self, n_filas=6, n_columnas=9):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.grid = np.zeros((n_filas, n_columnas), dtype=np.int8)
        centro = n_columnas // 2
        self._idx = indices_ventanas(n_filas, n_columnas)
        self._cols_centro = [centro - 1, centro, centro + 1]
        self._orden = sorted(range(n_columnas), key=lambda c: abs(c - centro))

    def colocar_ficha(self, color, columna):
        libres = np.flatnonzero(self.grid[:, columna] == 0)
        if libres.size == 0:
            return False
        self.grid[libres[-1], columna] = COLORES[color]
        return True

    def _ventanas(self):
        return self.grid.ravel()[self._idx]

    def verificar_victoria(self):
        v = self._ventanas()
        for color, k in COLORES.items():
            if (v == k).all(axis=1).any():
                return color
        return None

    def esta_lleno(self):
        return not (self.grid[0] == 0).any()

    def casillas_libres(self):
        return int((self.grid == 0).sum())

    def columnas_validas(self):
        return [c for c in self._orden if self.grid[0, c] == 0]

    def clonar(self):
        t = Tablero.__new__(Tablero)
        t.__dict__.update(self.__dict__)
        t.grid = self.grid.copy()
        return t

    def __str__(self):
        return "\n".join(
            "".join(SIMBOLOS[v] + ("" if v == 0 else " ") for v in fila) for fila in self.grid
        ) + "\n"

    def __repr__(self):
        return str(self)

    def evaluar_heuristica(self, color_jugador, heuristica="todas", pesos=None):
        P = pesos or PESOS_CONFIG_1
        if heuristica == "todas":
            activas = HEURISTICAS
        elif isinstance(heuristica, str):
            activas = [heuristica]
        else:
            activas = list(heuristica)

        yo = COLORES[color_jugador]
        rival = 3 - yo
        v = self._ventanas()
        propias = (v == yo).sum(axis=1)
        rivales = (v == rival).sum(axis=1)
        vacios = (v == 0).sum(axis=1)

        # Cada heurística queda en [-1, 1]; el signo ya refleja si favorece o perjudica
        valores = {}
        if "h1" in activas:
            centro = self.grid[:, self._cols_centro]
            valores["h1"] = ((centro == yo).sum() - (centro == rival).sum()) / centro.size
        if "h2" in activas:
            valores["h2"] = saturar(((propias == 4) & (vacios == 1)).sum())
        if "h3" in activas:
            valores["h3"] = -saturar(((rivales == 4) & (vacios == 1)).sum())
        if "h4" in activas:
            valores["h4"] = saturar(((propias == 3) & (vacios == 2)).sum())
        if "h5" in activas:
            valores["h5"] = -saturar(((rivales == 3) & (vacios == 2)).sum())

        total = sum(P[h] for h in activas)
        return ESCALA * sum(P[h] * valores[h] for h in activas) / total

In [ ]:
PROF_ARBOL = 2


class TimeoutException(Exception):
    pass


class Nodo:
    def __init__(self, estado, padre=None, accion=None, profundidad=0, tipo_nodo="MAX"):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.profundidad = profundidad
        self.tipo_nodo = tipo_nodo
        self.hijos = []
        self.valor_evaluado = None

    def expandir(self, color_max, color_min):
        self.hijos = []
        color = color_max if self.tipo_nodo == "MAX" else color_min
        siguiente = "MIN" if self.tipo_nodo == "MAX" else "MAX"
        for col in self.estado.columnas_validas():
            tablero = self.estado.clonar()
            tablero.colocar_ficha(color, col)
            self.hijos.append(Nodo(tablero, self, col, self.profundidad + 1, siguiente))
        return self.hijos

    def mostrar_arbol(self, max_prof=1, indent=""):
        if self.profundidad > max_prof:
            return
        accion = f"Jugada en Columna {self.accion}" if self.accion is not None else "Estado Inicial (Raíz)"
        valor = f" | Utilidad Minimax: {self.valor_evaluado:.1f}" if self.valor_evaluado is not None else ""
        print(f"{indent} Profundidad {self.profundidad} [{self.tipo_nodo}] - {accion}{valor}")
        for linea in str(self.estado).split("\n"):
            if linea.strip():
                print(f"{indent}  {linea}")
        for hijo in self.hijos:
            hijo.mostrar_arbol(max_prof, indent + "    ")


class Minimax_Solver:
    def __init__(self, player_color, max_time=2.0, heuristica="todas", pesos=None):
        self.player_color = player_color
        self.oponente_color = "Amarillo" if player_color == "Rojo" else "Rojo"
        self.max_time = max_time
        self.heuristica = heuristica
        self.pesos = pesos
        self.time_start = None
        self.ultimo_arbol_raiz = None
        self.nodos_expandidos = 0
        self.max_profundidad_alcanzada = 0
        self.tiempo_ultima_jugada = 0
        self.historial = []

    def encontrar_mejor_columna(self, tablero):
        self.time_start = time.time()
        self.nodos_expandidos = 0
        self.max_profundidad_alcanzada = 0

        validas = tablero.columnas_validas()
        if len(validas) <= 1:
            return validas[0] if validas else None

        mejor = validas[0]
        libres = tablero.casillas_libres()
        profundidad = 1

        while profundidad <= libres and time.time() - self.time_start < self.max_time:
            raiz = Nodo(tablero.clonar())
            try:
                utilidad, accion = self._maximize(raiz, -float("inf"), float("inf"), profundidad)
            except TimeoutException:
                break
            if accion is not None:
                mejor = accion
                raiz.valor_evaluado = utilidad
                self.ultimo_arbol_raiz = raiz
                self.max_profundidad_alcanzada = profundidad
            if abs(utilidad) > VICTORIA / 2:
                break
            profundidad += 1

        self.tiempo_ultima_jugada = time.time() - self.time_start
        self.historial.append({
            "nodos": self.nodos_expandidos,
            "prof": self.max_profundidad_alcanzada,
            "tiempo": self.tiempo_ultima_jugada,
        })
        return mejor

    def _valor_terminal(self, nodo, limite):
        ganador = nodo.estado.verificar_victoria()
        if ganador == self.player_color:
            return VICTORIA - nodo.profundidad
        if ganador == self.oponente_color:
            return -VICTORIA + nodo.profundidad
        if nodo.estado.esta_lleno():
            return 0
        if nodo.profundidad >= limite:
            return nodo.estado.evaluar_heuristica(self.player_color, self.heuristica, self.pesos)
        return None

    def _maximize(self, nodo, alpha, beta, limite):
        if time.time() - self.time_start >= self.max_time:
            raise TimeoutException()

        terminal = self._valor_terminal(nodo, limite)
        if terminal is not None:
            nodo.valor_evaluado = terminal
            return terminal, nodo.accion

        mejor_valor, mejor_accion = -float("inf"), None
        hijos = nodo.expandir(self.player_color, self.oponente_color)
        self.nodos_expandidos += len(hijos)

        for hijo in hijos:
            valor, _ = self._minimize(hijo, alpha, beta, limite)
            if valor > mejor_valor:
                mejor_valor, mejor_accion = valor, hijo.accion
            if mejor_valor >= beta:
                break
            alpha = max(alpha, mejor_valor)

        nodo.valor_evaluado = mejor_valor
        if nodo.profundidad >= PROF_ARBOL:
            nodo.hijos = []
        return mejor_valor, mejor_accion

    def _minimize(self, nodo, alpha, beta, limite):
        if time.time() - self.time_start >= self.max_time:
            raise TimeoutException()

        terminal = self._valor_terminal(nodo, limite)
        if terminal is not None:
            nodo.valor_evaluado = terminal
            return terminal, nodo.accion

        mejor_valor, mejor_accion = float("inf"), None
        hijos = nodo.expandir(self.player_color, self.oponente_color)
        self.nodos_expandidos += len(hijos)

        for hijo in hijos:
            valor, _ = self._maximize(hijo, alpha, beta, limite)
            if valor < mejor_valor:
                mejor_valor, mejor_accion = valor, hijo.accion
            if mejor_valor <= alpha:
                break
            beta = min(beta, mejor_valor)

        nodo.valor_evaluado = mejor_valor
        if nodo.profundidad >= PROF_ARBOL:
            nodo.hijos = []
        return mejor_valor, mejor_accion

In [ ]:
class Jugador:
    def __init__(self, nombre, color):
        self.nombre = nombre
        self.color = color

    def seleccionar_columna(self, tablero):
        raise NotImplementedError


class Humano(Jugador):
    def seleccionar_columna(self, tablero):
        validas = tablero.columnas_validas()
        while True:
            entrada = input(f"[{self.nombre}] Selecciona la columna: ").strip()
            if not entrada.isnumeric():
                print("Entrada inválida.")
            elif int(entrada) >= tablero.n_columnas:
                print(f"Fuera de rango (0 a {tablero.n_columnas - 1}).")
            elif int(entrada) not in validas:
                print("Columna llena.")
            else:
                return int(entrada)


class RandomBot(Jugador):
    def seleccionar_columna(self, tablero):
        return random.choice(tablero.columnas_validas())


class GreedyBot(Jugador):
    def seleccionar_columna(self, tablero):
        validas = tablero.columnas_validas()
        mejor_col, mejor_puntaje = random.choice(validas), -float("inf")
        for col in validas:
            temp = tablero.clonar()
            temp.colocar_ficha(self.color, col)
            if temp.verificar_victoria() == self.color:
                return col
            puntaje = temp.evaluar_heuristica(self.color)
            if puntaje > mejor_puntaje:
                mejor_col, mejor_puntaje = col, puntaje
        return mejor_col


class PeorBot(Jugador):
    def seleccionar_columna(self, tablero):
        validas = tablero.columnas_validas()
        peor_col, peor_puntaje = random.choice(validas), float("inf")
        for col in validas:
            temp = tablero.clonar()
            temp.colocar_ficha(self.color, col)
            puntaje = temp.evaluar_heuristica(self.color)
            if puntaje < peor_puntaje:
                peor_col, peor_puntaje = col, puntaje
        return peor_col


class MinimaxBot(Jugador):
    def __init__(self, nombre, color, max_time=2.0, heuristica="todas", pesos=None):
        super().__init__(nombre, color)
        self.solver = Minimax_Solver(color, max_time, heuristica, pesos)

    def seleccionar_columna(self, tablero):
        return self.solver.encontrar_mejor_columna(tablero)

    def reiniciar_metricas(self):
        self.solver.historial = []

    def metricas(self):
        h = self.solver.historial
        if not h:
            return {"nodos": 0, "prof_prom": 0, "prof_max": 0, "tiempo_total": 0, "tiempo_jugada": 0}
        return {
            "nodos": sum(m["nodos"] for m in h),
            "prof_prom": np.mean([m["prof"] for m in h]),
            "prof_max": max(m["prof"] for m in h),
            "tiempo_total": sum(m["tiempo"] for m in h),
            "tiempo_jugada": np.mean([m["tiempo"] for m in h]),
        }

    def imprimir_ultimo_arbol(self, max_profundidad=2):
        if self.solver.ultimo_arbol_raiz is None:
            print("Aún no se ha generado ningún árbol de búsqueda.")
            return
        print(f"\nÁrbol de búsqueda generado por {self.nombre} (última jugada)")
        self.solver.ultimo_arbol_raiz.mostrar_arbol(max_prof=max_profundidad)

In [ ]:
def jugar_partida(jugadores, mostrar_UI=True):
    tablero = Tablero()
    turno = 0
    while True:
        actual = jugadores[turno % 2]
        if mostrar_UI:
            limpiar()
            print(f"Turno de: {actual.nombre} ({actual.color})\n")
            print(tablero)

        tablero.colocar_ficha(actual.color, actual.seleccionar_columna(tablero))

        ganador = tablero.verificar_victoria()
        if ganador or tablero.esta_lleno():
            if mostrar_UI:
                limpiar()
                print(tablero)
                print(f"\nGanó {actual.nombre}" if ganador else "\nEmpate")
            return ganador, tablero
        turno += 1


def jugar_serie(principal, rival, n=1, alternar=True, mostrar_UI=False):
    if isinstance(principal, MinimaxBot):
        principal.reiniciar_metricas()

    res = {"V": 0, "E": 0, "D": 0}
    for i in range(n):
        invertir = alternar and i % 2 == 1
        orden = [rival, principal] if invertir else [principal, rival]
        ganador, _ = jugar_partida(orden, mostrar_UI)
        if ganador is None:
            res["E"] += 1
        elif ganador == principal.color:
            res["V"] += 1
        else:
            res["D"] += 1

        if mostrar_UI and n > 1:
            print(f"\nPartida {i + 1}/{n} | {principal.nombre} {res['V']} - {res['D']} {rival.nombre} | Empates: {res['E']}")
            if i < n - 1:
                input("Enter para la siguiente partida...")

    res["Puntos"] = 3 * res["V"] + res["E"]
    return res

In [ ]:
ORDEN_CONTRINCANTES = ["Random", "Greedy", "Peor", "Minimax igual", "Humano experto"]


def variantes_benchmark(tiempos=(1, 3, 10), t_base=1.0):
    base = {"max_time": t_base, "heuristica": "todas", "pesos": PESOS_CONFIG_1}
    variantes = [
        ("Pesos config 1", {**base}),
        ("Pesos config 2", {**base, "pesos": PESOS_CONFIG_2}),
    ]
    for k in range(1, 6):
        variantes.append((f"{k} heurística{'s' if k > 1 else ''}", {**base, "heuristica": HEURISTICAS[:k]}))
    for t in tiempos:
        variantes.append((f"max_time = {t}", {**base, "max_time": t}))
    return variantes


def crear_rival(nombre, kwargs):
    if nombre == "Random":
        return RandomBot("RND", "Amarillo")
    if nombre == "Greedy":
        return GreedyBot("GRD", "Amarillo")
    if nombre == "Peor":
        return PeorBot("WORST", "Amarillo")
    return MinimaxBot("Minimax-Rival", "Amarillo", **kwargs)


def construir_fila(variante, contrincante, ia, res):
    m = ia.metricas()
    if res["V"] > res["D"]:
        ganador = "Minimax"
    elif res["D"] > res["V"]:
        ganador = contrincante
    else:
        ganador = "Empate"
    return {
        "Variante": variante,
        "Contrincante": contrincante,
        "Ganador": ganador,
        "V-E-D": f"{res['V']}-{res['E']}-{res['D']}",
        "Puntos": res["Puntos"],
        "Nodos expandidos": m["nodos"],
        "Profundidad prom": round(m["prof_prom"], 2),
        "Profundidad máx": m["prof_max"],
        "Tiempo total (s)": round(m["tiempo_total"], 2),
        "Tiempo/jugada (s)": round(m["tiempo_jugada"], 3),
    }


class MenuInteractivo:
    def __init__(self):
        self.jugadores_partida = []
        self.filas = []

    def configurar_jugador(self, index, color):
        nombre = input(f"Nombre del Jugador {index} ({color}): ") or f"J{index}"
        print("1. Humano\n2. Minimax (IA)\n3. Greedy\n4. Random\n5. Peor Decisión")
        tipo = input("Selecciona (1-5): ").strip()

        if tipo == "2":
            tiempo = float(input("Tiempo máximo para la IA en segundos (ej: 2.0): ") or 2.0)
            print("Heurísticas: 1 Centro | 2 Ataque inminente | 3 Bloqueo crítico | 4 Construcción | 5 Prevención | 6 Todas")
            sel = input("Selecciona una o varias separadas por coma (ej: 1,3,5): ").replace(" ", "")
            elegidas = [f"h{x}" for x in sel.split(",") if x in list("12345")]
            heuristica = "todas" if "6" in sel or not elegidas else elegidas
            cfg = input("Pesos: 1 Balanceada | 2 Agresiva: ").strip()
            pesos = PESOS_CONFIG_2 if cfg == "2" else PESOS_CONFIG_1
            return MinimaxBot(nombre, color, tiempo, heuristica, pesos)
        if tipo == "3":
            return GreedyBot(nombre, color)
        if tipo == "4":
            return RandomBot(nombre, color)
        if tipo == "5":
            return PeorBot(nombre, color)
        return Humano(nombre, color)

    def jugar_manual(self):
        self.jugadores_partida = [self.configurar_jugador(1, "Rojo"), self.configurar_jugador(2, "Amarillo")]
        n = int(input("Número de partidas (1): ") or 1)
        res = jugar_serie(self.jugadores_partida[0], self.jugadores_partida[1], n, mostrar_UI=True)
        print(f"\nResultado final: {self.jugadores_partida[0].nombre} {res['V']} | "
              f"{self.jugadores_partida[1].nombre} {res['D']} | Empates {res['E']}")

    def _guardar_fila(self, fila):
        self.filas = [f for f in self.filas
                      if (f["Variante"], f["Contrincante"]) != (fila["Variante"], fila["Contrincante"])]
        self.filas.append(fila)

    def benchmark(self):
        partidas = int(input("Partidas por enfrentamiento (2): ") or 2)
        variantes = variantes_benchmark()
        rivales = ORDEN_CONTRINCANTES[:4]
        total, k = len(variantes) * len(rivales), 0

        for variante, kwargs in variantes:
            for rival_nombre in rivales:
                k += 1
                print(f"[{k}/{total}] {variante} vs {rival_nombre}")
                ia = MinimaxBot("Minimax", "Rojo", **kwargs)
                rival = crear_rival(rival_nombre, kwargs)
                res = jugar_serie(ia, rival, partidas)
                self._guardar_fila(construir_fila(variante, rival_nombre, ia, res))
        self.mostrar_resultados()

    def partida_humano(self):
        variantes = variantes_benchmark()
        for i, (nombre, _) in enumerate(variantes, 1):
            print(f"{i}. {nombre}")
        idx = int(input("Variante de Minimax a enfrentar: ")) - 1
        nombre, kwargs = variantes[idx]
        n = int(input("Número de partidas (2): ") or 2)

        ia = MinimaxBot("Minimax", "Rojo", **kwargs)
        humano = Humano("Humano", "Amarillo")
        res = jugar_serie(ia, humano, n, mostrar_UI=True)
        self._guardar_fila(construir_fila(nombre, "Humano experto", ia, res))
        print(f"\nRegistrado: {nombre} vs Humano experto -> V-E-D {res['V']}-{res['E']}-{res['D']}")

    def mostrar_resultados(self):
        if not self.filas:
            print("Aún no hay resultados.")
            return
        df = pd.DataFrame(self.filas)
        variantes = list(dict.fromkeys(df["Variante"]))
        rivales = [r for r in ORDEN_CONTRINCANTES if r in set(df["Contrincante"])]

        def tabla(metrica):
            return df.pivot(index="Variante", columns="Contrincante", values=metrica).reindex(
                index=variantes, columns=rivales)

        display(df)
        for metrica in ["V-E-D", "Puntos", "Nodos expandidos", "Profundidad prom", "Tiempo/jugada (s)"]:
            print(f"\n{metrica}")
            display(tabla(metrica))

        fig, ejes = plt.subplots(2, 2, figsize=(16, 9))
        for ax, metrica in zip(ejes.ravel(), ["Puntos", "Nodos expandidos", "Profundidad prom", "Tiempo/jugada (s)"]):
            tabla(metrica).astype(float).plot(kind="bar", ax=ax, logy=(metrica == "Nodos expandidos"))
            ax.set_title(metrica)
            ax.set_xlabel("")
            ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()
        df.to_csv("benchmark_resultados.csv", index=False)

    def menu_principal(self):
        while True:
            print("\n1. Jugar partida(s)\n2. Ejecutar benchmark\n3. Humano vs Minimax (se registra en el benchmark)"
                  "\n4. Mostrar resultados\n0. Salir")
            opcion = input("Elige una opción: ").strip()
            if opcion == "1":
                self.jugar_manual()
            elif opcion == "2":
                self.benchmark()
            elif opcion == "3":
                self.partida_humano()
            elif opcion == "4":
                self.mostrar_resultados()
            elif opcion == "0":
                break

In [ ]:
juego = MenuInteractivo()
juego.menu_principal()

In [ ]:
for jugador in juego.jugadores_partida:
    if isinstance(jugador, MinimaxBot):
        jugador.imprimir_ultimo_arbol(max_profundidad=PROF_ARBOL)